In this notebook we implement pybdshadow for calculating building's shadowing in bologna's city for adressing a Urban Thermal Adaptability index 

In [ ]:
!pip install pybdshadow

In [ ]:
import pandas as pd
import geopandas as gpd
import pybdshadow
import matplotlib.pyplot as plt
import json
import time
import os
from tqdm import tqdm
from pyproj import CRS
import numpy as np

import warnings
warnings.filterwarnings("ignore", message="CRS not set for some of the concatenation inputs")
warnings.simplefilter(action='ignore', category=FutureWarning)


In [ ]:
# computing shadows 
# THIS BLOCK TAKES AROUND 1H20' TO FINISH. THE OUTPUTS ARE STORED IN THE GOOGLE DRIVE AS "shadows.tar.gz"

num_parts = 20 # number of files in clean_results (the number comes from 2.4.2.3.trees-vector-preparation.ipynb)
input_folder = "../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/trees-vectors-clean/"
output_folder = '../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/trees-shadows/'
os.makedirs(output_folder, exist_ok=True)

date = pd.to_datetime('2024-08-09 09:58:10').tz_localize('UTC')

for j in tqdm(range(num_parts)):
    part_file = os.path.join(input_folder, f"part{j+1}_clean.geojson")
    if not os.path.exists(part_file):
        continue
        
    trees = gpd.read_file(part_file)
    trees = trees.to_crs(CRS.from_epsg(4326)) #this was done before in the previous notebook with clean_results but just in case
    trees = pybdshadow.bd_preprocess(trees)

    # --------------------------
    # Divide each one in 5 parts for avoiding problems with RAM
    subparts = np.array_split(trees, 5)
    shadow_part = []

    for k, part in enumerate(subparts):
        try:
            start = time.time()
            shadow_subpart = pybdshadow.bdshadow_sunlight(part, date, roof=False, include_building=True)
            print(f"Processing time part{j} subpart{k+1}: {round(time.time() - start, 2)} seconds")
            shadow_part.append(shadow_subpart)
        except Exception as e:
            print(f"Error en part{j} subpart{k+1}: {e}")
        time.sleep(10)
    
    # join results and save
    shadows = gpd.GeoDataFrame(pd.concat(shadow_part, ignore_index=True), crs=shadow_part[0].crs)
    # dissolve
    union = shadows.union_all()
    shadows = gpd.GeoDataFrame(geometry=[union], crs=shadows.crs).explode(index_parts=False)

    output_path = os.path.join(output_folder, f"part{j}_shadow.geojson")
    #shadows.set_crs("EPSG:7791", inplace=True)
    shadows.to_file(output_path, driver="GeoJSON")


In [ ]:
# join all the results in one dataframe

input_folder = '../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/trees-shadows/'
total_shadows = []

for j in tqdm(range(num_parts)):
    
    part_file = os.path.join(input_folder, f"part{j}_shadow.geojson") #correjir a j++1 aca y anterior bloque
    if not os.path.exists(part_file):
        continue
    shadow_part = gpd.read_file(part_file)
    total_shadows.append(shadow_part)

    # Unir y guardar
    shadows = gpd.GeoDataFrame(pd.concat(total_shadows, ignore_index=True), crs=total_shadows[0].crs)


shadows.to_file('../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/shadows_trees_2024-08-09_095810', driver="GeoJSON")